In [ ]:
#Functions to analyze above indicators
def trend_break(df, trend_col, direction_col):
    """
    This function identifies the time when a trend line is broken.
    Args:
        df (pd.DataFrame): Dataframe containing historical data of a security.
        trend_col (str): Name of the column containing the trend line values.
        direction_col (str): Name of the column to store the direction of the trend (increasing or decreasing).

    Returns:
        A list of tuples containing the date and direction of the trend (increasing or decreasing) when the trend line was broken.
    """
    trend = df[trend_col]
    prev_trend = trend[0]
    direction = ""
    trend_breaks = []
    for i in range(1, len(df)):
        if trend[i] > prev_trend:
            if direction != "increasing":
                direction = "increasing"
        elif trend[i] < prev_trend:
            if direction != "decreasing":
                direction = "decreasing"
        else:
            continue

        if (trend[i] > trend[i-1] and trend[i] > trend[i+1]) or (trend[i] < trend[i-1] and trend[i] < trend[i+1]):
            trend_breaks.append((df.iloc[i]['Date'], direction))
        prev_trend = trend[i]
    return trend_breaks

In [ ]:
#we can make this function dynamic and apply it to any previously created indicator. We can create a new function that takes the data for the indicator, as well as the trend 
# line breakpoints and the column names for the signal line and histogram, and returns a DataFrame with information on each trend line break.
def trend_line_break_accuracy(data, trend_breaks, signal_col, hist_col):
    """
    Calculate the accuracy of the given indicator at each trend line break.

    Args:
        data (pd.DataFrame): DataFrame containing the indicator data
        trend_breaks (pd.DataFrame): DataFrame containing trend line breaks
        signal_col (str): Name of the column containing the signal line data
        hist_col (str): Name of the column containing the histogram data

    Returns:
        pd.DataFrame: DataFrame containing information on each trend line break
    """
    results = []
    for i, row in trend_breaks.iterrows():
        start_date = row['start_date']
        end_date = row['end_date']
        trend_direction = row['trend_direction']
        signal_start = data.loc[data['Date'] == start_date, signal_col].values[0]
        signal_end = data.loc[data['Date'] == end_date, signal_col].values[0]
        hist_start = data.loc[data['Date'] == start_date, hist_col].values[0]
        hist_end = data.loc[data['Date'] == end_date, hist_col].values[0]
        if trend_direction == 'upward':
            if signal_end > signal_start and hist_end > hist_start:
                accuracy = 1
            elif signal_end > signal_start or hist_end > hist_start:
                accuracy = 0.5
            else:
                accuracy = 0
        else:
            if signal_end < signal_start and hist_end < hist_start:
                accuracy = 1
            elif signal_end < signal_start or hist_end < hist_start:
                accuracy = 0.5
            else:
                accuracy = 0
        results.append({
            'start_date': start_date,
            'end_date': end_date,
            'trend_direction': trend_direction,
            'accuracy': accuracy
        })
    return pd.DataFrame(results)

In [ ]:
def feature_engineering(df):
    # Add a column that measures the length of the trend
    df['trend_length'] = (pd.to_datetime(df['end_date']) - pd.to_datetime(df['start_date'])).dt.days
    
    # Add a column that measures the distance between the start and end price of the trend
    df['price_distance'] = df['end_price'] - df['start_price']
    
    # Add a column that measures the difference in the signal line values between the start and end of the trend
    df['signal_difference'] = df['end_signal'] - df['start_signal']
    
    # Add a column that measures the difference in the histogram values between the start and end of the trend
    df['hist_difference'] = df['end_hist'] - df['start_hist']
    
    # Add a column that measures the accuracy of the trend line break as a percentage
    df['accuracy_percentage'] = df['accuracy'] * 100
    
    # Add a column that measures the strength of the trend, which is the product of trend length and price distance
    df['trend_strength'] = df['trend_length'] * df['price_distance']
    
    return df
#This function takes in a dataframe that contains the output of trend_line_break_accuracy. It adds several new columns to the dataframe, including columns that measure the length of 
# the trend, the distance between the start and end price of the trend, the difference in the signal line values between the start and end of the trend, the difference in the 
# histogram values between the start and end of the trend, the accuracy of the trend line break as a percentage, and the strength of the trend.

#Here's an example script that would do feature engineering on the accuracy of each indicator when analyzed with trend_line_break_accuracy:
from TechnicalIndicators import TechnicalIndicators

# Initialize TechnicalIndicators object
ti = TechnicalIndicators('AAPL', '2010-01-01', '2021-12-31')

# Get all indicator data
all_data = ti.get_all_indicators()

# Get trend line breaks
trend_breaks = trend_break(all_data, 'Close', 'trend_direction')

# Dictionary to store accuracy data
accuracy_data = {}

# Loop through each indicator and calculate accuracy data
for k, v in TechnicalIndicators.INDICATOR_FUNCTIONS.items():
    indicator_data = getattr(ti.data.ta, v)()
    trend_line_data = trend_line_break_accuracy(indicator_data, trend_breaks, f"{k}_signal", f"{k}_hist")
    accuracy_data[k] = trend_line_data['accuracy'].mean()

# Convert accuracy data to a DataFrame
accuracy_df = pd.DataFrame.from_dict(accuracy_data, orient='index', columns=['accuracy'])

# Feature engineering
accuracy_df['above_mean'] = accuracy_df['accuracy'] > accuracy_df['accuracy'].mean()
accuracy_df['rank'] = accuracy_df['accuracy'].rank(ascending=False)
accuracy_df = accuracy_df.sort_values('rank')

# Print accuracy data with features
print(accuracy_df)
#In this script, we first initialize a TechnicalIndicators object to get all indicator data. We then calculate trend line breaks using the trend_break function and store the 
# results in a DataFrame. Next, we loop through each indicator and calculate accuracy data using the trend_line_break_accuracy function. We store the accuracy data in a dictionary.

#We then convert the accuracy data to a DataFrame and perform feature engineering on the accuracy scores. In this example, we add two features: above_mean, which indicates whether 
# an indicator's accuracy is above the mean accuracy score across all indicators, and rank, which ranks the indicators by accuracy score.

#Finally, we sort the DataFrame by the rank column and print the accuracy data with features.

In [ ]:
#a Random Forest Classifier can be used to analyze the accuracy of each indicator when they are analyzed with trend_line_break_accuracy. The classifier can be trained on the 
# features generated by the feature engineering process, and then used to predict the accuracy of each indicator. The predicted accuracies can then be compared with the actual 
# accuracies to evaluate the performance of the classifier.

#Here's an example of how you could implement this:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Get the data with the trend line break accuracies
ti = TechnicalIndicators('AAPL', '2020-01-01', '2022-01-01')
all_data = ti.get_all_indicators()
trend_breaks = trend_break(all_data, 'Super_trend', 'trend')

# Do feature engineering on the trend line break accuracies
features = feature_engineering(trend_breaks)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, all_data[ti.INDICATOR_FUNCTIONS.keys()], test_size=0.2, random_state=42)

# Train a Random Forest Classifier
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf.predict(X_test)

# Evaluate the performance of the classifier
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

#Keras Model

from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam

# Load data
data = pd.read_csv('accuracy_features.csv')

# Split data into train and test sets
train_data, test_data = train_test_split(data, test_size=0.2)

# Separate features and target
X_train = train_data.drop(['accuracy'], axis=1).values
y_train = train_data['accuracy'].values
X_test = test_data.drop(['accuracy'], axis=1).values
y_test = test_data['accuracy'].values

# Define the neural network model
model = Sequential()
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32)

# Evaluate the model on test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test loss: {loss}')
print(f'Test accuracy: {accuracy}')

#This code first loads the feature engineered data from a CSV file, then splits it into train and test sets. It then defines a neural network model with two hidden layers and 
# one output layer. The model is trained on the train data and evaluated on the test data. The final accuracy of the model is printed.


In [ ]:
ticker = 'AAPL'
start_date = '2019-01-01'
end_date = '2021-01-01'
strike_price = 150
expiry_date = '2021-03-19'

fair_value_price, option_price, percent_diff = option_analysis(ticker, start_date, end_date, strike_price, expiry_date)
print(f"Fair value price: {fair_value_price}")
print(f"Option price: {option_price}")
print(f"Percent difference: {percent_diff}")

# Note that this script assumes that you have access to a Yahoo Finance API key in order to download the historical stock and option data. It also assumes that you have the 
# necessary permissions to access the CBOE Options API.